# Quality Control with scLucid

This notebook demonstrates quality control workflows for single-cell RNA-seq data.

In [ ]:
import scanpy as sc
import scLucid
from scLucid.qc import QCWorkflowConfig, run_standard_qc

# Set global config
from scLucid.config import set_config
set_config(n_jobs=4, verbosity=1)

## Load Data

In [ ]:
# Load your data
adata = sc.read_h5ad("data/raw_counts.h5ad")

print(f"Cells: {adata.n_obs}")
print(f"Genes: {adata.n_vars}")

## Configure QC Workflow

In [ ]:
qc_config = QCWorkflowConfig(
    # QC thresholds
    thresholds={
        "min_genes": 200,
        "max_genes": 5000,
        "pc_mt": 20.0
    },
    # Doublet detection
    doublet={
        "method": "scrublet",
        "expected_doublet_rate": 0.06
    },
    # Save results
    save_dir="results/qc"
)

## Run Standard QC Workflow

In [ ]:
adata_qc = run_standard_qc(adata, config=qc_config)

print(f"Original cells: {adata.n_obs}")
print(f"Filtered cells: {adata_qc.n_obs}")
print(f"Removed: {(1 - adata_qc.n_obs/adata.n_obs)*100:.1f}%")

## Visualize QC Metrics

In [ ]:
# QC plots were saved to results/qc/
# Let's visualize the filtered data
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Before filtering
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], 
              ax=axes[0], show=False)
axes[0].set_title('Before QC')

# After filtering
sc.pl.violin(adata_qc, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], 
              ax=axes[1], show=False)
axes[1].set_title('After QC')

# Doublet scores
sc.pl.violin(adata_qc, ['doublet_score'], ax=axes[2], show=False)
axes[2].set_title('Doublet Scores')

plt.tight_layout()
plt.savefig('results/qc/comparison.pdf')

## Save QC Results

In [ ]:
# Save filtered data
adata_qc.write('results/qc_filtered.h5ad')

# Save config for reproducibility
qc_config.to_json_file('results/qc_config.json')